# v-modes & DOF Normalization via `ts_ofc` StateEstimator (v1)

**Author:** Aaron Roodman
**Date Created:** 2026-08-13
**Last Modified:** 2026-08-13
**Status:** Complete
**Keywords:** AOS, OFC, v-modes, sensitivity matrix, normalization, StateEstimator

## Description

Builds the **DOF-vs-v-mode** matrix for the `22_12` correction scheme (22 DOF, 12 v-modes)
directly from `ts_ofc`'s `StateEstimator` — **no inline SVD, no `ts_intrinsic_wavefront`**.
It also documents the geometric normalization used to define the v-modes and the units of
the normalized DOF coordinate.

Key functionality:
1. Configure `OFCData` (v13 config) restricted to the 22-DOF standard set via `comp_dof_idx`.
2. Build the DOF-per-v-mode matrix from `StateEstimator.get_dofs_from_vmodes`.
3. Plot it in both **normalized** (review-standard) and **physical** (native-unit) form.

**Output:** A two-panel figure of the DOF composition of each v-mode.

**Based on:** `ts_ofc` `StateEstimator`; normalization per Eqs. 9-11 of
[Manuel et al. 2024, ApJ 974, 108](https://ui.adsabs.harvard.edu/abs/2024ApJ...974..108M)
and `ts_ofc/scripts/generate_normalization_weights.py`.

## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-08-13 | Aaron Roodman | Initial version (ts_ofc-only) |

## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Background: v-modes & the geometric normalization](#background)
4. [Helper Functions](#functions)
5. [OFCData & StateEstimator setup](#data)
6. [Analysis: DOF-vs-v-mode matrices](#analysis)
7. [Results & Plots](#results)

<a id='params'></a>
## Parameters

In [ ]:
# ============================================================
# Parameters — All configurable values collected here
# ============================================================
import os

INSTRUMENT = "lsst"

# v13 config dir — this is where range0.5_fwhm-0.15.yaml lives.
CONFIG_DIR = os.path.join(os.environ["TS_CONFIG_MTTCS_DIR"], "MTAOS", "v13", "ofc")

# 22_12 scheme: 22 DOF (M2 hex 5 + Cam hex 5 + M1M3 B1-7 + M2 B1-5), keep 12 v-modes.
N_M1M3_BEND = 7
N_M2_BEND   = 5
N_KEEP      = 12

<a id='setup'></a>
## Setup & Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import SymLogNorm

# ts_ofc only — no ts_intrinsic_wavefront, no rubin-work code
from lsst.ts.ofc import OFCData
from lsst.ts.ofc.state_estimator import StateEstimator

# Minimal, self-contained plotting defaults (keeps the notebook portable)
plt.rcParams.update({
    "figure.dpi": 110,
    "font.size": 10,
    "axes.titlesize": 10,
    "image.cmap": "RdBu_r",
})

<a id='background'></a>
## Background: v-modes & the geometric normalization

The OFC state estimator builds v-modes from the **double-Zernike sensitivity matrix** $S$
(focal-field Zernike × pupil Zernike × DOF), which maps degrees of freedom (DOF) to the
wavefront. Before the SVD, each DOF column is rescaled by a normalization weight $w_i$:

$$ S_{\text{scaled}} = S\,\mathrm{diag}(w), \qquad S_{\text{scaled}} = U\,\Sigma\,V^\top $$

The **v-modes** are the columns of $V$ — unit vectors in **normalized-DOF space**. Physical
DOF are recovered by $\mathrm{DOF}_{\text{phys}} = \mathrm{diag}(w)\,V\,a$, which is exactly
what `StateEstimator.get_dofs_from_vmodes` returns. (`get_vmodes_from_dofs` is the inverse
map, used to project measured DOF onto v-modes.)

### The normalization weight

Each weight is a **geometric combination** (product of powers) of a *range* factor and an
*FWHM* factor:

$$ w_i = r_i^{\alpha}\,f_i^{\beta}, \qquad \alpha = +0.5,\ \ \beta = -0.5
   \;\;\Longrightarrow\;\; w_i = \sqrt{r_i/f_i} $$

- **$r_i$ — range factor:** the available travel of DOF $i$ (hexapod stroke `rb_stroke`;
  for bending modes, `force_range/20 / max|BendModeToForce|`, an amplitude range). Native
  units: µm (piston/decenter), arcsec (tilt), µm (bending amplitude). The `range0.5` file
  uses **half** the full stroke.
- **$f_i$ — FWHM factor:** the PSF degradation *per unit DOF*,
  `convertZernikesToPsfWidth(S)` aggregated over field points and Zernikes
  (quadrature / RMS). Units: arcsec-FWHM per unit DOF.

These are computed in `ts_ofc/scripts/generate_normalization_weights.py`; the exact
$(\alpha,\beta)$ live in the `metadata:` block of the weights YAML —
here `range0.5_fwhm-0.15` $\Rightarrow \alpha=0.5,\ \beta=-0.5$.

### Units

| quantity | expression | units |
|---|---|---|
| range factor $r_i$ | stroke | µm (or arcsec) |
| FWHM factor $f_i$ | FWHM per unit DOF | arcsec / (DOF unit) |
| weight $w_i$ | $\sqrt{r_i/f_i}$ | µm / $\sqrt{\text{arcsec}}$ |
| normalized DOF $x_i = \mathrm{phys}_i / w_i$ |  | **$\sqrt{\text{arcsec}}$** (uniform, all DOF) |

The normalized coordinate has the **same unit $\sqrt{\text{arcsec}}$ for every DOF** —
hexapod µm, hexapod arcsec, and bending µm alike — which is what puts them on a common
footing.

### Physical meaning

Regrouping,

$$ x_i = \mathrm{phys}_i\,\sqrt{f_i/r_i}
      = \sqrt{\underbrace{(\mathrm{phys}_i\,f_i)}_{\text{FWHM produced}}\;\cdot\;
              \underbrace{(\mathrm{phys}_i/r_i)}_{\text{fraction of range used}}} $$

so each normalized DOF is the **geometric mean of the PSF blur it produces and the fraction
of its stroke it consumes** — balancing *don't blur the image* against *don't burn actuator
range*. That is why the v-modes are reviewed in this normalized space.

> **Direction / reciprocal note.** The *weight* $w_i=\sqrt{r_i/f_i}$ has range on top. The
> factor that *normalizes* a physical DOF, $1/w_i=\sqrt{f_i/r_i}=\sqrt{\text{FWHM}}/\sqrt{\text{range}}$,
> is the reciprocal. With the default $\alpha=\beta=1$ instead, $w_i=r_i f_i$ has units
> µm·arcsec (as the YAML header states); the $\tfrac12,-\tfrac12$ choice gives the
> $\sqrt{\text{arcsec}}$ normalized coordinate above.

<a id='functions'></a>
## Helper Functions

In [ ]:
def dof_labels(n_m1m3=N_M1M3_BEND, n_m2=N_M2_BEND):
    """Row labels for the 22-DOF standard set."""
    return (
        ["M2Hex dz", "M2Hex dx", "M2Hex dy", "M2Hex rx", "M2Hex ry"]
        + ["CamHex dz", "CamHex dx", "CamHex dy", "CamHex rx", "CamHex ry"]
        + [f"M1M3 B{i}" for i in range(1, n_m1m3 + 1)]
        + [f"M2 B{i}" for i in range(1, n_m2 + 1)]
    )


def dof_vmode_matrices(se, n_keep):
    """DOF-vs-v-mode matrices built ONLY from StateEstimator (no inline SVD).

    Column m is the physical DOF vector produced by a unit amplitude in v-mode m,
    i.e. ``get_dofs_from_vmodes(e_m) == N @ V[:, m]``.

    Parameters
    ----------
    se : StateEstimator
        State estimator whose SVD (U, S, Vh, normalization_matrix) is already built.
    n_keep : int
        Number of v-modes to keep (truncation).

    Returns
    -------
    M_phys : np.ndarray, shape (n_dof, n_keep)
        DOF per unit v-mode in PHYSICAL units (= N * V).
    V_norm : np.ndarray, shape (n_dof, n_keep)
        DOF per unit v-mode in NORMALIZED (dimensionless) units (= V).
    """
    se.truncate_index = n_keep  # keep exactly n_keep modes
    M_phys = np.column_stack(
        [se.get_dofs_from_vmodes(np.eye(n_keep)[m]) for m in range(n_keep)]
    )
    Ndiag = np.diag(se.normalization_matrix)  # per-DOF normalization weights w_i
    V_norm = M_phys / Ndiag[:, None]          # divide out w_i -> normalized v-modes
    return M_phys, V_norm


def dof_units(n_m1m3=N_M1M3_BEND, n_m2=N_M2_BEND, inverse=False):
    """Unit string for each DOF row (same order as dof_labels).

    inverse=False -> the DOF's own unit (um / arcsec); used for M_phys, the
                     DOF-per-v-mode matrix.
    inverse=True  -> inverse unit (1/um / 1/arcsec); used for the v-mode-per-DOF
                     coefficients in the "v1 = C1*DOF1 + ..." formula.
    """
    hexa = ["um", "um", "um", "arcsec", "arcsec"]  # dz, dx, dy, rx, ry
    base = hexa + hexa + ["um"] * n_m1m3 + ["um"] * n_m2
    return [f"1/{u}" for u in base] if inverse else base


def vmode_formula(coeff, rank_by, labels, unit_strs, mode=0, n_terms=5,
                  coeff_desc="coefficient"):
    """Print v-mode `mode` as a formula in terms of the DOF.

    Terms are ranked by importance in the NORMALIZED space (|rank_by|, the common
    footing).  The printed coefficients come from `coeff`, with units `unit_strs`.
    """
    order = np.argsort(np.abs(rank_by[:, mode]))[::-1][:n_terms]

    one_line = "  ".join(
        f"{coeff[i, mode]:+.4g} {unit_strs[i]}*[{labels[i]}]" for i in order
    )
    print(f"v{mode+1} = {one_line}\n")

    print(f"top {n_terms} terms by normalized weight ({coeff_desc} | normalized):")
    for i in order:
        print(f"  {coeff[i, mode]:+12.4g} {unit_strs[i]:<8s} * {labels[i]:<10s}"
              f"   (norm {rank_by[i, mode]:+.3f})")

    frac = np.sum(rank_by[order, mode] ** 2) / np.sum(rank_by[:, mode] ** 2)
    print(f"\nthese {n_terms} terms capture {100*frac:.1f}% of the "
          f"normalized mode power")

<a id='data'></a>
## OFCData & StateEstimator setup

In [ ]:
# --- OFCData with the v13 config (where range0.5_fwhm-0.15.yaml lives) ---
ofc = OFCData(INSTRUMENT, config_dir=CONFIG_DIR)
ofc.configure_controller()                    # loads normalization_weights + truncation
await ofc.configure_instrument(INSTRUMENT)    # loads the sensitivity matrix (async)

# --- Restrict to the 22-DOF standard set via comp_dof_idx ---
ofc.comp_dof_idx = dict(
    m2HexPos=np.ones(5, bool),
    camHexPos=np.ones(5, bool),
    M1M3Bend=(np.arange(20) < N_M1M3_BEND),   # first 7 M1M3 bending modes
    M2Bend=(np.arange(20) < N_M2_BEND),       # first 5 M2 bending modes
)

# --- StateEstimator performs the SVD internally (U, S, Vh, normalization_matrix) ---
se = StateEstimator(ofc)
print(f"used DOF: {len(ofc.dof_idx)}   v-modes available: {se.Vh.shape[0]}   keeping: {N_KEEP}")

<a id='analysis'></a>
## Analysis: DOF-vs-v-mode matrices

In [ ]:
M_phys, V_norm = dof_vmode_matrices(se, N_KEEP)
labels = dof_labels()
modes = [f"v{m+1}" for m in range(N_KEEP)]

print("M_phys (physical DOF per v-mode) shape:", M_phys.shape)
print("V_norm (normalized DOF per v-mode) shape:", V_norm.shape)

# v-mode-per-unit-DOF coefficients (the "v1 = C1*DOF1 + ..." direction), built
# purely from StateEstimator.get_vmodes_from_dofs. C[j, m] = amplitude of v-mode m
# produced by moving used-DOF j by ONE unit (1 um or 1 arcsec) -> INVERSE DOF units.
C_perDOF = np.zeros((len(ofc.dof_idx), N_KEEP))
for j, gidx in enumerate(ofc.dof_idx):
    e = np.zeros(50)
    e[int(gidx)] = 1.0
    C_perDOF[j, :] = se.get_vmodes_from_dofs(e)[:N_KEEP]

# Consistency: C_perDOF is exactly V_norm / w, the reciprocal of the M_phys weighting.
assert np.allclose(C_perDOF, V_norm / np.diag(se.normalization_matrix)[:, None])
print("C_perDOF (v-mode per unit DOF, inverse units) shape:", C_perDOF.shape)

# Sanity checks (all use StateEstimator only):
# 1. V_norm columns are unit vectors in normalized-DOF space (right singular vectors).
col_norms = np.linalg.norm(V_norm, axis=0)
print("V_norm column norms (should be ~1):", np.round(col_norms, 6))

# 2. get_vmodes_from_dofs is the inverse of get_dofs_from_vmodes on the kept subspace.
a = np.zeros(N_KEEP); a[3] = 1.0                    # a unit v-mode amplitude vector
dof_full = np.zeros(50)
dof_full[ofc.dof_idx] = se.get_dofs_from_vmodes(a)  # -> physical DOF (length 50)
a_back = se.get_vmodes_from_dofs(dof_full)[:N_KEEP]  # -> back to v-mode amplitudes
print("round-trip max |a - a_back|:", np.max(np.abs(a - a_back)))

<a id='results'></a>
## Results & Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 8))

# Left: normalized v-modes (unit vectors in normalized-DOF space) — review-standard
im0 = axes[0].imshow(V_norm, aspect="auto", vmin=-1, vmax=1)
axes[0].set_title(
    "v-modes in NORMALIZED DOF space\n"
    r"coefficient units: $\sqrt{\mathrm{arcsec}}$ (all DOF on common footing)"
)
fig.colorbar(im0, ax=axes[0], shrink=0.8,
             label=r"normalized DOF coefficient  [$\sqrt{\mathrm{arcsec}}$]")

# Right: PHYSICAL units on a symmetric-log (positive/negative log) scale, so all
# coefficients — which span several orders of magnitude — are visible at once.
plim = np.abs(M_phys).max()
nz = np.abs(M_phys)[np.abs(M_phys) > 0]
linthresh = max(1e-4, np.percentile(nz, 5))   # linear near 0 below this; adjust to taste
norm = SymLogNorm(linthresh=linthresh, vmin=-plim, vmax=plim, base=10)
im1 = axes[1].imshow(M_phys, aspect="auto", norm=norm)
axes[1].set_title(
    "v-modes in PHYSICAL DOF units (symmetric log)\n"
    "hexapod: piston/decenter [µm], tilt [arcsec];  bending: amplitude [µm]"
)
fig.colorbar(im1, ax=axes[1], shrink=0.8,
             label="physical DOF per unit v-mode  [µm or arcsec], symlog")

for ax in axes:
    ax.set_xticks(range(N_KEEP)); ax.set_xticklabels(modes)
    ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels)
    ax.set_xlabel("v-mode")

fig.suptitle("22_12 scheme — DOF composition per v-mode (ts_ofc StateEstimator)", fontsize=12)
fig.tight_layout()
plt.show()

### v-mode 1 as a formula in the DOF

Here v-mode 1 is written as a scalar function of the DOF,
$v_1 = \sum_i C_i\,\mathrm{DOF}_i$, so plugging in DOF values (µm, arcsec) returns the
v-mode amplitude. The coefficients $C_i$ are therefore in **inverse DOF units** (1/µm for
the µm DOF here). They come from `StateEstimator.get_vmodes_from_dofs` — the projection
(DOF → v-mode) direction — and equal $V_{\text{norm}}/w$.

Terms are ranked by importance in the **normalized** space (`|V_norm|`, the common footing).

> This is the **reciprocal** of the right-hand plot panel, which shows the *DOF-per-v-mode*
> amounts $M_{\text{phys}} = w\,V_{\text{norm}}$ (how far each DOF moves to realize one unit
> of the v-mode, in µm/arcsec). Large µm there ⟷ small 1/µm here.

In [ ]:
units_inv = dof_units(inverse=True)
vmode_formula(C_perDOF, V_norm, labels, units_inv, mode=0, n_terms=5,
              coeff_desc="1/DOF coefficient")